# Task-05: Food Recognition & Calorie Estimation — Walkthrough

Demonstrates the enterprise pipeline: synthetic data generation -> feature extraction -> classification -> calorie lookup.

Dataset schema: [Kaggle Food-101](https://www.kaggle.com/dansbecker/food-101). Real data is drop-in replaceable in `data/raw/<class_name>/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.utils.config_loader import load_config
from src.utils.logger import get_logger

## 1. Load configuration
All paths are resolved to absolute, project-root-anchored paths.

In [ ]:
cfg = load_config()
logger = get_logger('notebook', cfg)
print('Raw data dir:', cfg['data']['raw_dir'])
print('Model backend:', cfg['model']['backend'])
print('Classes:', cfg['synthetic']['class_names'][:cfg['synthetic']['num_classes']])

## 2. Generate synthetic dataset
Matches the Food-101 folder-per-class schema. Skips classes that already have enough images.

In [ ]:
from src.data.synthetic_generator import generate_synthetic_dataset
generate_synthetic_dataset(cfg, logger)

## 3. Load and split dataset

In [ ]:
from src.data.data_loader import load_dataset, split_dataset
X, y, class_names = load_dataset(cfg, logger)
X_train, X_test, y_train, y_test = split_dataset(X, y, cfg, logger)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

### Preview sample images per class

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, cname in zip(axes.flat, class_names[:10]):
    idx = np.where(y_train == cname)[0][0]
    ax.imshow(X_train[idx])
    ax.set_title(cname, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/predictions/sample_images.png', dpi=100)
plt.close()
print('Saved sample grid to outputs/predictions/sample_images.png')

## 4. Feature extraction (fit on train, transform test)
HOG texture descriptors + per-channel color histograms. Scaler fit only on training data to prevent leakage.

In [ ]:
from src.features.feature_extractor import FoodFeatureExtractor
fe = FoodFeatureExtractor(cfg, logger)
X_train_feat = fe.fit_transform(X_train)
X_test_feat = fe.transform(X_test)
print('Feature dim:', X_train_feat.shape[1])

## 5. Train classifier
Backend selected via config (`sklearn_rf` today; swappable for a CNN backend behind the same interface).

In [ ]:
from src.models.sklearn_classifier import SklearnFoodClassifier
clf = SklearnFoodClassifier(cfg, logger)
clf.fit(X_train_feat, y_train)
y_pred = clf.predict(X_test_feat)

## 6. Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
print(f'Accuracy: {acc:.4f}')
print(f'F1 (macro): {f1:.4f}')

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=class_names)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=90)
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('../outputs/predictions/confusion_matrix.png', dpi=100)
plt.close()
print('Saved confusion matrix to outputs/predictions/confusion_matrix.png')

## 7. Calorie estimation
Map each predicted class to an estimated calorie value via lookup table.

In [ ]:
from src.models.calorie_estimator import CalorieEstimator
ce = CalorieEstimator(cfg, logger)
sample_preds = list(y_pred[:8])
sample_calories = ce.estimate_batch(sample_preds)
for cls, cal in zip(sample_preds, sample_calories):
    print(f'{cls:15s} -> {cal} kcal/100g')

## 8. Full pipeline (equivalent to `python main.py`)
Orchestrates all steps above and writes artifacts to `outputs/`.

In [ ]:
from src.pipeline.pipeline import run_pipeline
metrics = run_pipeline(cfg, logger)
print('Final accuracy:', metrics['accuracy'])
print('Final F1 (macro):', metrics['f1_macro'])

## Summary
- Model, metrics, run summary, and predictions (with calorie estimates) are saved under `outputs/`
- Real Food-101 images can replace synthetic data by dropping them into `data/raw/<class_name>/` and setting `synthetic.enabled: false`
- Swapping to a CNN backend (TensorFlow/PyTorch) requires only a new class implementing `BaseFoodClassifier` — no pipeline changes